# Vehicle Price Prediction

An exploratory course project on used-car prices: data preparation, exploratory analysis, regression, regularization and model diagnostics.

Run the code cells in order. Outputs are cleared in this public copy. The dataset is downloaded from its original Kaggle source; it is not included in this repository.

**Evaluation note:** the historical workflow compares multiple models using the same test split and makes some data-cleaning choices before splitting. Treat scores as exploratory results, not an unbiased estimate of performance on new data. Cross-validation here also follows preprocessing fitted on the training partition rather than a fully fold-contained pipeline. A future evaluation should use a separate untouched holdout and fit preprocessing within each cross-validation fold.


# Step 1: Import Libraries & Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn import metrics
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from statsmodels.stats.diagnostic import normal_ad
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson
from scipy import stats
from scipy.special import inv_boxcox
import kagglehub
import os

path = kagglehub.dataset_download("nehalbirla/vehicle-dataset-from-cardekho")
print("Path to dataset files:", path)


# Step 2: Dataset Overview

In [ ]:
csv_path = os.path.join(path, "car data.csv")
df = pd.read_csv(csv_path)

df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe(include='number')

In [ ]:
df.describe(include='object')

## Summary

- **Dataset size and overall quality**  
  - The dataset contains **301 observations and 9 columns**, which is a relatively small but manageable size for regression modelling and experimentation.  
  - From `df.info()`, all columns have **301 non-null values**, indicating there are no missing values and the overall data quality is good. No additional imputation is required at this stage.

- **Feature types and basic interpretation**  
  - There are **5 numerical features** (`int64` / `float64`): `Year`, `Selling_Price`, `Present_Price`, `Kms_Driven`, and `Owner`. These can be directly used for statistical analysis and as inputs to regression models.  
  - There are **4 categorical features** (`object`): `Car_Name`, `Fuel_Type`, `Seller_Type`, and `Transmission`. These are suitable for frequency tables and bar plots, and will need to be encoded (e.g., one-hot encoding) before being used in most machine learning models.  
  - `Selling_Price` is a natural choice for the **target variable (y)**, while the remaining columns can serve as explanatory features (X).

- **Potential data issues and next steps**  
  - `Car_Name` likely has many unique values and behaves more like a text label; its direct impact on price may be limited. It may be useful to simplify it (e.g., extract brand only) or drop it depending on later analysis.  
  - The `Owner` variable is expected to be highly imbalanced (most cars are first-owner), which should be kept in mind when interpreting model coefficients or feature importance.  
  - As a next step, we can explore the numerical features using histograms and boxplots to check their distributions and potential outliers, and use bar plots for the categorical features to understand how fuel type, seller type and transmission relate to selling price trends.


# Step 3: Data Preparation

In this step, we prepare the dataset for modelling. We will:
- Select a meaningful subset of features.
- Transform the `Year` column into a more informative `Age` feature.
- Detect and analyse outliers.
- Check for missing values and duplicates.
- Decide which observations to remove to make the later regression model more robust.


## Step 3.1: Feature Subset Selection

First, we inspect the `Car_Name` column to understand how many distinct car names exist in the dataset. A very high cardinality for a categorical feature can make encoding inefficient and may not bring much additional information to the model.


In [ ]:
# Number of unique car names
df['Car_Name'].nunique()


The dataset contains **98 unique car names**, which is a large number of categories compared to the total sample size (301 observations). Encoding this column would introduce many dummy variables with limited additional information, so we drop `Car_Name` from the feature set.


In [ ]:
# Drop the high-cardinality categorical feature
df.drop('Car_Name', axis=1, inplace=True)
df.head()


## Step 3.2: Feature Transformation – From `Year` to `Age`

The `Year` column represents the model year of each car. For price modelling, **car age** is usually more interpretable and closer to how value depreciates over time. We compute `Age` as the difference between one year after the maximum model year in the dataset and the car's model year, then drop the original `Year` column.


In [ ]:
origin_year = df["Year"].max() + 1

df.insert(0, "Age", origin_year - df["Year"])
df.drop('Year', axis=1, inplace=True)

df.head()


## Step 3.3: Outlier Detection

Outliers are observations that deviate strongly from the general pattern of the data. Because linear regression is sensitive to extreme values, it is important to detect these points before fitting the model. We first visualise each feature with boxplots and then formally identify outliers using the Interquartile Range (IQR) rule.


In [ ]:
# Boxplots for numerical and categorical features
sns.set_style('darkgrid')
colors = ['#0055ff', '#ff7000', '#23bf00']
sns.set_palette(sns.color_palette(colors))

ordered_cols = np.concatenate([
    df.select_dtypes(exclude='object').columns.values,
    df.select_dtypes(include='object').columns.values
])

fig, ax = plt.subplots(2, 4, figsize=(15, 7), dpi=100)

for i, col in enumerate(ordered_cols):
    x = i // 4
    y = i % 4

    if i < 5:
        sns.boxplot(data=df, y=col, ax=ax[x, y])
        ax[x, y].yaxis.label.set_size(15)
    else:
        sns.boxplot(data=df, x=col, y='Selling_Price', ax=ax[x, y])
        ax[x, y].xaxis.label.set_size(15)
        ax[x, y].yaxis.label.set_size(15)

plt.tight_layout()
plt.show()


In [ ]:
# Identify outliers using the IQR rule
outliers_indexes = []
target = 'Selling_Price'

# 1) Outliers within each category of the categorical features
for col in df.select_dtypes(include='object').columns:
    for cat in df[col].unique():
        df_cat = df[df[col] == cat]
        q1 = df_cat[target].quantile(0.25)
        q3 = df_cat[target].quantile(0.75)
        iqr = q3 - q1
        maximum = q3 + 1.5 * iqr
        minimum = q1 - 1.5 * iqr
        outlier_samples = df_cat[(df_cat[target] < minimum) | (df_cat[target] > maximum)]
        outliers_indexes.extend(outlier_samples.index.tolist())

# 2) Outliers based on the numerical features themselves
for col in df.select_dtypes(exclude='object').columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    maximum = q3 + 1.5 * iqr
    minimum = q1 - 1.5 * iqr
    outlier_samples = df[(df[col] < minimum) | (df[col] > maximum)]
    outliers_indexes.extend(outlier_samples.index.tolist())

outliers_indexes = list(set(outliers_indexes))
print('{} outliers were identified, whose indices are:\n\n{}'.format(len(outliers_indexes), outliers_indexes))


We identify **outliers using the IQR rule** for both categorical-condition slices of `Selling_Price` and for each numerical feature. In total, 38 observations are flagged as potential outliers. However, we do **not** automatically drop all of them; instead, we further examine which of these points create unrealistic or extreme patterns in the data.


In [ ]:
# Label outliers and define which ones to remove

df1 = df.copy()
df1['label'] = 'Normal'
df1.loc[outliers_indexes, 'label'] = 'Outlier'

removing_indexes = []
removing_indexes.extend(df1[df1[target] > 33].index)
removing_indexes.extend(df1[df1['Kms_Driven'] > 400000].index)
df1.loc[removing_indexes, 'label'] = 'Removing'

# Visualise normal points, outliers and points to be removed
features = df.columns.drop(target)
colors = ['#0055ff', '#ff7000', '#23bf00']
sns.set_palette(sns.color_palette(colors))

fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(15, 12), dpi=200)

for i in range(len(features)):
    x = i // 3
    y = i % 3
    sns.scatterplot(data=df1, x=features[i], y=target, hue='label', ax=ax[x, y])
    ax[x, y].set_title('{} vs. {}'.format(target, features[i]), size=15)
    ax[x, y].set_xlabel(features[i], size=12)
    ax[x, y].set_ylabel(target, size=12)
    ax[x, y].grid()

ax[2, 1].axis('off')
ax[2, 2].axis('off')
plt.tight_layout()
plt.show()

removing_indexes = list(set(removing_indexes))
removing_indexes


We find that only **two observations** (indices 196 and 86) behave very differently from the overall pattern across several scatter plots. Because linear regression is highly sensitive to such extreme points, we decide to remove only these clearly anomalous samples and keep the rest of the data.


## Step 3.4: Missing Values

We check whether any feature contains missing values using `df.isnull().sum()`.


In [ ]:
df.isnull().sum()


The result shows that **all features have zero missing values**, so we do not need to apply any imputation technique in this project.


## Step 3.5: Duplicate Observations

Next, we check for duplicated rows to understand whether there are repeated cars with exactly the same attributes.


In [ ]:
# Inspect duplicated rows (if any)
df[df.duplicated(keep=False)]


The duplicated rows correspond to cars that share exactly the same attributes (age, price, mileage, fuel type, seller type, transmission and owner). Since this is still a realistic situation in the market, we **keep** these duplicates and do not remove them.


## Step 3.6: Dropping Selected Outliers

Finally, we remove only the clearly extreme observations identified earlier (indices 196 and 86) and create a cleaned dataset `df1` for subsequent modelling.


In [ ]:
# Drop the selected outliers and reset index
df1 = df.copy()
df1.drop(removing_indexes, inplace=True)
df1.reset_index(drop=True, inplace=True)

df1.shape


# Step 4: Enhanced Exploratory Data Analysis (EDA)

In this step, we perform a comprehensive exploratory data analysis to understand the data distribution, relationships between features, and prepare insights for modeling. We will:
- Analyze categorical variables univariately
- Analyze numerical variables univariately with distribution statistics
- Examine correlations between numerical features
- Explore bivariate relationships between target and numerical features
- Explore bivariate relationships between target and categorical features
- Perform multivariate and interaction analysis
- Summarize EDA findings and provide modeling insights


## Step 4.1: Categorical Variables Univariate Analysis

**Objective**: Examine the frequency distribution of categorical features.

**Action**: Use `sns.countplot` to visualize the count distribution for `Fuel_Type`, `Seller_Type`, and `Transmission`.

**Conclusion**: Summarize the proportion of each category.


In [ ]:
# Categorical variables univariate analysis
categorical_features = ['Fuel_Type', 'Seller_Type', 'Transmission']

fig, axes = plt.subplots(1, 3, figsize=(15, 5), dpi=100)
sns.set_style('darkgrid')

for i, feature in enumerate(categorical_features):
    sns.countplot(data=df1, x=feature, ax=axes[i], palette='viridis')
    axes[i].set_title(f'Count Distribution: {feature}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel(feature, fontsize=12)
    axes[i].set_ylabel('Count', fontsize=12)
    
    # Add percentage labels on bars
    total = len(df1)
    for p in axes[i].patches:
        height = p.get_height()
        axes[i].text(p.get_x() + p.get_width()/2., height,
                    f'{int(height)}\n({height/total*100:.1f}%)',
                    ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

# Display frequency tables
print("\n" + "="*60)
print("CATEGORICAL VARIABLES FREQUENCY SUMMARY")
print("="*60)
for feature in categorical_features:
    print(f"\n{feature}:")
    freq = df1[feature].value_counts()
    prop = df1[feature].value_counts(normalize=True) * 100
    summary = pd.DataFrame({
        'Count': freq,
        'Percentage': prop
    })
    print(summary.to_string())


**Conclusion**: The categorical variables show the following distribution:
- **Fuel_Type**: Most cars use Petrol, followed by Diesel and CNG.
- **Seller_Type**: The majority are sold by Dealers, with fewer Individual sellers.
- **Transmission**: Manual transmission dominates, with Automatic being less common.

These distributions indicate class imbalance, which should be considered when interpreting model coefficients and feature importance.


## Step 4.2: Numerical Variables Univariate Analysis

**Objective**: Examine the distribution shape of numerical features.

**Action**: Use `sns.histplot` to create histograms with KDE, and calculate mean, median, standard deviation, skewness, and kurtosis.

**Conclusion**: Confirm that all numerical variables are right-skewed, providing quantitative evidence for subsequent data transformation.


In [ ]:
# Numerical variables univariate analysis
numerical_features = df1.select_dtypes(exclude='object').columns.tolist()

n_features = len(numerical_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5*n_rows), dpi=100)
axes = axes.flatten() if n_features > 1 else [axes]

for i, feature in enumerate(numerical_features):
    sns.histplot(data=df1, x=feature, kde=True, ax=axes[i], bins=30, color='#0055ff')
    axes[i].set_title(f'Distribution: {feature}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel(feature, fontsize=12)
    axes[i].set_ylabel('Frequency', fontsize=12)
    axes[i].grid(True, alpha=0.3)

# Hide unused subplots
for i in range(n_features, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("NUMERICAL VARIABLES DISTRIBUTION STATISTICS")
print("="*80)

stats_dict = {
    'Feature': [],
    'Mean': [],
    'Median': [],
    'Std Dev': [],
    'Skewness': [],
    'Kurtosis': []
}

for feature in numerical_features:
    data = df1[feature]
    stats_dict['Feature'].append(feature)
    stats_dict['Mean'].append(data.mean())
    stats_dict['Median'].append(data.median())
    stats_dict['Std Dev'].append(data.std())
    stats_dict['Skewness'].append(stats.skew(data))
    stats_dict['Kurtosis'].append(stats.kurtosis(data))

stats_df = pd.DataFrame(stats_dict)
print("\n", stats_df.to_string(index=False))

# Interpretation
print("\n" + "="*80)
print("INTERPRETATION")
print("="*80)
print("\nSkewness Interpretation:")
print("  - Skewness > 0: Right-skewed (positive skew)")
print("  - Skewness < 0: Left-skewed (negative skew)")
print("  - Skewness ≈ 0: Approximately symmetric")
print("\nKurtosis Interpretation:")
print("  - Kurtosis > 0: Heavy-tailed (leptokurtic)")
print("  - Kurtosis < 0: Light-tailed (platykurtic)")
print("  - Kurtosis ≈ 0: Normal-like tails (mesokurtic)")


**Conclusion**: All numerical variables exhibit **right-skewed distributions** (positive skewness), with the following observations:
- **Selling_Price** and **Present_Price** show strong right skewness, indicating most cars have lower prices with a few high-value outliers.
- **Kms_Driven** is right-skewed, suggesting most cars have lower mileage.
- **Age** and **Owner** also show positive skewness.

This quantitative evidence supports the need for **logarithmic or Box-Cox transformation** to normalize the distributions and improve model performance, especially for linear regression models that assume normality.


## Step 4.3: Numerical Feature Correlation Analysis

**Objective**: Quantify the linear relationship strength between all numerical variables and identify multicollinearity.

**Action**:
- Calculate correlation coefficient matrix between all numerical features and the target variable `Selling_Price`.
- Use `sns.heatmap` to visualize the correlation heatmap.

**Conclusion**: Highlight the strong positive correlation between `Selling_Price` and `Present_Price`, as well as other feature correlations (e.g., `Age` and `Year` have been addressed in Step 3).


In [ ]:
correlation_matrix = df1[numerical_features].corr()

# Plot correlation heatmap
plt.figure(figsize=(10, 8), dpi=100)
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool), k=1)  # Mask upper triangle
sns.heatmap(correlation_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, mask=mask,
            vmin=-1, vmax=1, annot_kws={'size': 11, 'weight': 'bold'})
plt.title('Correlation Matrix: Numerical Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Display correlation with target variable
print("\n" + "="*60)
print("CORRELATION WITH TARGET VARIABLE (Selling_Price)")
print("="*60)
target_corr = correlation_matrix['Selling_Price'].drop('Selling_Price').sort_values(ascending=False)
target_corr_df = pd.DataFrame({
    'Feature': target_corr.index,
    'Correlation': target_corr.values
})
print("\n", target_corr_df.to_string(index=False))

# Identify strong correlations (potential multicollinearity)
print("\n" + "="*60)
print("POTENTIAL MULTICOLLINEARITY (|correlation| > 0.7)")
print("="*60)
high_corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        corr_val = correlation_matrix.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr_pairs.append({
                'Feature 1': correlation_matrix.columns[i],
                'Feature 2': correlation_matrix.columns[j],
                'Correlation': corr_val
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs)
    print("\n", high_corr_df.to_string(index=False))
else:
    print("\nNo strong multicollinearity detected (|correlation| > 0.7).")


**Conclusion**: 
- **Selling_Price** shows a **strong positive correlation with Present_Price** (correlation = 0.884), indicating that the current market price is the most important predictor of selling price.
- **Age** shows a **moderate negative correlation with Selling_Price** (correlation = -0.275), confirming that older cars tend to have lower selling prices.
- **Kms_Driven** shows a **weak positive correlation** (correlation = 0.063), while **Owner** shows a **weak negative correlation** (correlation = -0.091) with the target variable. Both relationships are very weak and may not be statistically significant.
- No severe multicollinearity issues are detected among the predictor variables (all pairwise correlations between predictors < 0.7), which is good for linear regression modeling.


## Step 4.4: Target vs. Numerical Features Bivariate Analysis

**Objective**: Visualize the relationship between the target variable and numerical features.

**Action**: Use `plt.scatter` to create scatter plots of `Selling_Price` vs. `Present_Price`, `Kms_Driven`, `Age`, and `Owner`.

**Conclusion**: Confirm the positive relationship with `Present_Price` and inverse relationships with `Kms_Driven`, `Age`, and `Owner`.


In [ ]:
# Target vs. numerical features bivariate analysis
numerical_predictors = [col for col in numerical_features if col != 'Selling_Price']

n_preds = len(numerical_predictors)
n_cols = 2
n_rows = (n_preds + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 5*n_rows), dpi=100)
axes = axes.flatten() if n_preds > 1 else [axes]

for i, feature in enumerate(numerical_predictors):
    axes[i].scatter(df1[feature], df1['Selling_Price'], alpha=0.6, s=50, color='#0055ff')
    
    # Add trend line
    z = np.polyfit(df1[feature], df1['Selling_Price'], 1)
    p = np.poly1d(z)
    axes[i].plot(df1[feature], p(df1[feature]), "r--", alpha=0.8, linewidth=2, label='Trend line')
    
    axes[i].set_xlabel(feature, fontsize=12)
    axes[i].set_ylabel('Selling_Price', fontsize=12)
    axes[i].set_title(f'Selling_Price vs. {feature}', fontsize=14, fontweight='bold')
    axes[i].grid(True, alpha=0.3)
    axes[i].legend()
    
    # Add correlation coefficient
    corr_coef = df1[feature].corr(df1['Selling_Price'])
    axes[i].text(0.05, 0.95, f'Correlation: {corr_coef:.3f}', 
                transform=axes[i].transAxes, fontsize=11,
                verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Hide unused subplots
for i in range(n_preds, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()


**Conclusion**:
- **Selling_Price vs. Present_Price**: Shows a **strong positive linear relationship** (correlation = 0.884), confirming that higher current market prices correspond to higher selling prices. This is the most predictive feature.
- **Selling_Price vs. Kms_Driven**: Shows a **weak positive relationship** (correlation = 0.063), indicating a very slight tendency for cars with higher mileage to have slightly higher selling prices, though the relationship is extremely weak and may not be meaningful.
- **Selling_Price vs. Age**: Shows a **moderate negative relationship** (correlation = -0.275), confirming that older cars (higher age) have lower selling prices, which aligns with depreciation patterns.
- **Selling_Price vs. Owner**: Shows a **weak negative relationship** (correlation = -0.091), suggesting that cars with more previous owners tend to have slightly lower prices, though the effect is minimal.

These relationships support the use of linear regression, though transformations may be needed to improve linearity.


## Step 4.5: Target vs. Categorical Features Bivariate Analysis

**Objective**: Compare the distribution differences of the target variable across different categories.

**Action**:
- Use **Box Plots** or **Violin Plots** to visualize `Selling_Price` vs. `Fuel_Type`, `Seller_Type`, and `Transmission`.
- (Optional) Compare the median values across groups.

**Conclusion**: Confirm that Diesel/Automatic/Dealer categories correspond to higher median selling prices and wider price distributions.


In [ ]:
# Target vs. categorical features bivariate analysis
fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=100)

for i, feature in enumerate(categorical_features):
    # Box plot
    sns.boxplot(data=df1, x=feature, y='Selling_Price', ax=axes[i], palette='Set2')
    axes[i].set_title(f'Selling_Price by {feature}', fontsize=14, fontweight='bold')
    axes[i].set_xlabel(feature, fontsize=12)
    axes[i].set_ylabel('Selling_Price', fontsize=12)
    axes[i].grid(True, alpha=0.3, axis='y')
    
    # Add median labels
    medians = df1.groupby(feature)['Selling_Price'].median()
    for j, (cat, median_val) in enumerate(medians.items()):
        axes[i].text(j, median_val, f'Med: {median_val:.2f}', 
                    ha='center', va='bottom', fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='yellow', alpha=0.7))

plt.tight_layout()
plt.show()

# Violin plots for more detailed distribution
fig, axes = plt.subplots(1, 3, figsize=(16, 5), dpi=100)

for i, feature in enumerate(categorical_features):
    sns.violinplot(data=df1, x=feature, y='Selling_Price', ax=axes[i], palette='Set2', inner='box')
    axes[i].set_title(f'Selling_Price Distribution by {feature} (Violin Plot)', fontsize=14, fontweight='bold')
    axes[i].set_xlabel(feature, fontsize=12)
    axes[i].set_ylabel('Selling_Price', fontsize=12)
    axes[i].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Statistical summary by category
print("\n" + "="*80)
print("STATISTICAL SUMMARY: Selling_Price by Categorical Features")
print("="*80)

for feature in categorical_features:
    print(f"\n{feature}:")
    summary = df1.groupby(feature)['Selling_Price'].agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    summary.columns = ['Count', 'Mean', 'Median', 'Std Dev', 'Min', 'Max']
    print(summary.to_string())


**Conclusion**:
- **Fuel_Type**: 
  - **Diesel** cars have the **highest median selling price** (7.75), followed by CNG (3.10), with Petrol having the lowest (2.675).
  - Diesel cars also show a wider price distribution (Std Dev = 6.47), indicating more variability compared to Petrol (Std Dev = 3.14) and CNG (Std Dev = 0.21).
  
- **Seller_Type**:
  - **Dealer** sales show **significantly higher median selling prices** (5.25) compared to Individual sellers (0.52).
  - Dealer sales have a much wider price range (Std Dev = 4.73 vs 1.65), suggesting they handle both budget and premium vehicles, while Individual sellers primarily deal with lower-priced cars.
  
- **Transmission**:
  - **Automatic** transmission cars have a **significantly higher median selling price** (5.85) than Manual transmission cars (3.25).
  - Automatic cars show a wider price distribution (Std Dev = 8.75 vs 3.25), indicating they span a broader price range, likely due to the association with premium vehicles.

These findings confirm that categorical features have meaningful predictive power and should be included in the model with appropriate encoding (One-Hot Encoding or Label Encoding).


## Step 4.6: Multivariate and Interaction Analysis

**Objective**: Discover complex interaction effects between features.

**Action**: Use `sns.lmplot` to plot `Selling_Price` vs. `Present_Price`, faceted by `Seller_Type` and `Transmission` (col/row), and colored by `Fuel_Type` (hue).

**Conclusion**:
- Confirm the segmented patterns discovered in Step 4.5 (e.g., Individual sellers only sell Petrol cars).
- Focus on whether the price growth slope from `Present_Price` differs across groups (e.g., Dealer vs. Individual), revealing interaction effects.


In [ ]:
# Multivariate and interaction analysis
# Plot 1: Selling_Price vs. Present_Price, faceted by Seller_Type and Transmission, colored by Fuel_Type
g1 = sns.lmplot(data=df1, x='Present_Price', y='Selling_Price', 
                col='Seller_Type', row='Transmission', hue='Fuel_Type',
                height=5, aspect=1.2, palette='Set2', scatter_kws={'alpha': 0.6, 's': 50},
                line_kws={'linewidth': 2})
g1.fig.suptitle('Selling_Price vs. Present_Price: Interaction Analysis', 
                fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Plot 2: Selling_Price vs. Present_Price, faceted by Fuel_Type, colored by Transmission
g2 = sns.lmplot(data=df1, x='Present_Price', y='Selling_Price', 
                col='Fuel_Type', hue='Transmission',
                height=5, aspect=1.2, palette='Set1', scatter_kws={'alpha': 0.6, 's': 50},
                line_kws={'linewidth': 2})
g2.fig.suptitle('Selling_Price vs. Present_Price by Fuel_Type and Transmission', 
                fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("INTERACTION EFFECT ANALYSIS: Slope of Present_Price by Category Groups")
print("="*80)

from sklearn.linear_model import LinearRegression

interaction_results = []

for seller in df1['Seller_Type'].unique():
    for trans in df1['Transmission'].unique():
        subset = df1[(df1['Seller_Type'] == seller) & (df1['Transmission'] == trans)]
        if len(subset) > 2:  # Need at least 3 points for regression
            X = subset[['Present_Price']].values
            y = subset['Selling_Price'].values
            model = LinearRegression()
            model.fit(X, y)
            slope = model.coef_[0]
            intercept = model.intercept_
            r2 = model.score(X, y)
            
            interaction_results.append({
                'Seller_Type': seller,
                'Transmission': trans,
                'Count': len(subset),
                'Slope': slope,
                'Intercept': intercept,
                'R²': r2
            })

if interaction_results:
    interaction_df = pd.DataFrame(interaction_results)
    print("\n", interaction_df.to_string(index=False))
    
    print("\n" + "-"*80)
    print("INTERPRETATION:")
    print("-"*80)
    print("Different slopes across groups indicate interaction effects.")
    print("A steeper slope means Present_Price has a stronger impact on Selling_Price for that group.")
else:
    print("\nInsufficient data for interaction analysis.")


**Conclusion**:
- **Segmented Patterns**: 
  - The faceted plots reveal that **Individual sellers primarily deal with Petrol cars**, while Dealers handle a more diverse mix including Diesel and CNG vehicles.
  - This confirms the segmented market structure discovered in Step 4.5.
  
- **Interaction Effects**:
  - The **slope of Present_Price** (rate of price increase) may differ across Seller_Type and Transmission combinations.
  - For example, the relationship between Present_Price and Selling_Price might be stronger (steeper slope) for Dealer sales or Automatic transmission cars.
  - These interaction effects suggest that including interaction terms (e.g., `Present_Price × Seller_Type`) in the model could improve predictive performance.
  
- **Modeling Implication**: Consider adding interaction terms or using tree-based models that can naturally capture these non-linear relationships.


## Step 4.7: EDA Summary and Modeling Insights

**Objective**: Transform all findings into concrete recommendations for subsequent data preprocessing and model selection.

**Summary**:
- **High predictive features**: Present_Price (strong positive correlation).
- **Features requiring transformation**: Selling_Price and all numerical features (due to severe right-skewness, recommend logarithmic or Box-Cox transformation).
- **Features requiring encoding**: All categorical features (Fuel_Type, Seller_Type, Transmission) need One-Hot Encoding or Label Encoding.
- **Processed features**: Confirmed deletion of Car_Name, creation of Age, and removal of key outliers.


In [ ]:
# EDA Summary and Modeling Insights
print("="*80)
print("EDA SUMMARY AND MODELING INSIGHTS")
print("="*80)

print("\n" + "1. HIGH PREDICTIVE FEATURES")
print("-" * 80)
print("   • Present_Price: Strong positive correlation with Selling_Price (r > 0.8)")
print("     → This is the most important predictor and should be prioritized in feature selection.")

print("\n" + "2. FEATURES REQUIRING TRANSFORMATION")
print("-" * 80)
print("   • Target Variable (Selling_Price): Right-skewed distribution")
print("   • All Numerical Features: Right-skewed distributions confirmed by skewness > 0")
print("   → RECOMMENDATION: Apply logarithmic or Box-Cox transformation to:")
for feature in numerical_features:
    skew_val = stats.skew(df1[feature])
    print(f"     - {feature} (skewness: {skew_val:.3f})")
print("   → This will normalize distributions and improve linear regression assumptions.")

print("\n" + "3. FEATURES REQUIRING ENCODING")
print("-" * 80)
print("   • Categorical Features that need encoding:")
for feature in categorical_features:
    n_categories = df1[feature].nunique()
    print(f"     - {feature}: {n_categories} categories")
    print(f"       Categories: {', '.join(df1[feature].unique())}")
print("   → RECOMMENDATION: Use One-Hot Encoding for all categorical features")
print("   → Alternative: Label Encoding if ordinal relationships exist (not applicable here)")

print("\n" + "4. PROCESSED FEATURES")
print("-" * 80)
print("   ✓ Car_Name: Removed (high cardinality: 98 unique values)")
print("   ✓ Year: Transformed to Age (more interpretable for price modeling)")
print("   ✓ Outliers: Removed 2 extreme observations (indices 196, 86)")
print("   ✓ Dataset size: Reduced from 301 to 299 observations")

print("\n" + "5. MULTICOLLINEARITY ASSESSMENT")
print("-" * 80)
print("   ✓ No severe multicollinearity detected (all pairwise correlations < 0.7)")
print("   → All numerical features can be included in the model without VIF concerns.")

print("\n" + "6. INTERACTION EFFECTS")
print("-" * 80)
print("   • Evidence of interaction effects between:")
print("     - Present_Price × Seller_Type")
print("     - Present_Price × Transmission")
print("     - Fuel_Type × Seller_Type (market segmentation)")
print("   → RECOMMENDATION: Consider adding interaction terms or using non-linear models")

print("\n" + "7. CLASS IMBALANCE")
print("-" * 80)
for feature in categorical_features:
    proportions = df1[feature].value_counts(normalize=True)
    max_prop = proportions.max()
    if max_prop > 0.7:
        print(f"   • {feature}: Significant class imbalance (dominant class: {max_prop:.1%})")
print("   → Consider this when interpreting model coefficients and feature importance")

print("\n" + "8. NEXT STEPS FOR MODELING")
print("-" * 80)
print("   1. Apply logarithmic transformation to target and numerical features")
print("   2. Apply One-Hot Encoding to categorical features")
print("   3. Split data into training and testing sets")
print("   4. Fit linear regression model with transformed features")
print("   5. Evaluate model performance and consider interaction terms if needed")
print("   6. Apply regularization (Ridge/Lasso) if overfitting occurs")

print("\n" + "="*80)


# Step 5: Categorical Variables Encoding

After completing all EDA based on original features (Step 4.1 to 4.6), we now need to encode categorical features to enable comprehensive correlation analysis (Step 6) and final modeling.

**Objective**: Convert nominal categorical variables into numerical form for comprehensive quantitative analysis and model training.

**Action**: Apply dummy variable encoding (Dummy Encoding) to all categorical features (`Fuel_Type`, `Seller_Type`, `Transmission`) using `drop_first=True` to avoid multicollinearity.


In [ ]:
# Categorical variables encoding
CatCols = ['Fuel_Type', 'Seller_Type', 'Transmission']

df_encoded = df1.copy()

# Perform dummy variable encoding and drop the first category to avoid multicollinearity
df_encoded = pd.get_dummies(df_encoded, columns=CatCols, drop_first=True)

print("Encoded dataset shape:", df_encoded.shape)
print("\nEncoded dataset first 5 rows:")
print(df_encoded.head())

print("\n" + "="*80)
print("ENCODED FEATURES SUMMARY")
print("="*80)
print("\nOriginal categorical features:", CatCols)
print("\nNew encoded features:")
encoded_features = [col for col in df_encoded.columns if any(cat_col in col for cat_col in CatCols)]
print(encoded_features)
print(f"\nTotal features after encoding: {len(df_encoded.columns)}")
print(f"Target variable: Selling_Price")


**Conclusion**: The categorical features have been successfully encoded using dummy variable encoding with `drop_first=True`:
- **Fuel_Type**: Encoded into two binary features (`Fuel_Type_Diesel`, `Fuel_Type_Petrol`), with `CNG` dropped as the reference category.
- **Seller_Type**: Encoded into one binary feature (`Seller_Type_Individual`), with `Dealer` dropped as the reference category.
- **Transmission**: Encoded into one binary feature (`Transmission_Manual`), with `Automatic` dropped as the reference category.

The encoded dataset now contains **9 features** (5 numerical + 4 encoded categorical features) plus the target variable `Selling_Price`. This encoding approach avoids the dummy variable trap (perfect multicollinearity) while preserving all information needed for modeling. The reference categories (CNG, Dealer, Automatic) are implicitly represented when all encoded features are 0.


# Step 6: Comprehensive Correlation Analysis

Now we use the encoded dataset `df_encoded` to perform comprehensive correlation analysis, which serves as a key step in quantifying feature relationships in the entire workflow.

**Objective**: Quantify the linear relationships between all features (including numerical and encoded categorical features) and the target variable, and identify multicollinearity among independent features.

**Action**:
- Calculate correlation coefficient matrix for all features.
- Use `sns.heatmap` to visualize the correlation heatmap.
- Extract correlations with the target variable `Selling_Price`.


In [ ]:
# Comprehensive correlation analysis
target = 'Selling_Price'

corr_matrix = df_encoded.corr()

cmap = sns.diverging_palette(125, 28, s=100, l=65, sep=50, as_cmap=True)

plt.figure(figsize=(14, 12), dpi=100)
sns.heatmap(corr_matrix, 
            annot=True, 
            fmt=".2f",  # Display 2 decimal places
            cmap=cmap,
            center=0,
            square=True,
            linewidths=0.5,  # Line width for separation
            cbar_kws={"shrink": 0.8},
            vmin=-1, vmax=1)
plt.title('Comprehensive Correlation Heatmap: All Features', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

# Extract correlations with Selling_Price
selling_price_corr = corr_matrix[target].sort_values(ascending=False)

print("\n" + "="*80)
print("CORRELATION WITH TARGET VARIABLE (Selling_Price)")
print("="*80)
print("\n", selling_price_corr.to_string())

# Identify strong correlations (potential multicollinearity among predictors)
print("\n" + "="*80)
print("POTENTIAL MULTICOLLINEARITY AMONG PREDICTORS (|correlation| > 0.7)")
print("="*80)

predictor_corr = corr_matrix.drop(index=[target], columns=[target])
high_corr_pairs = []

for i in range(len(predictor_corr.columns)):
    for j in range(i+1, len(predictor_corr.columns)):
        corr_val = predictor_corr.iloc[i, j]
        if abs(corr_val) > 0.7:
            high_corr_pairs.append({
                'Feature 1': predictor_corr.columns[i],
                'Feature 2': predictor_corr.columns[j],
                'Correlation': corr_val
            })

if high_corr_pairs:
    high_corr_df = pd.DataFrame(high_corr_pairs)
    print("\n", high_corr_df.to_string(index=False))
    print("\n⚠️  WARNING: Strong multicollinearity detected among predictors!")
    print("   Consider removing one of the highly correlated features or using regularization.")
else:
    print("\n✓ No strong multicollinearity detected among predictors (|correlation| > 0.7).")

# Summary of top correlations with target
print("\n" + "="*80)
print("TOP FEATURES CORRELATED WITH Selling_Price")
print("="*80)
top_positive = selling_price_corr[selling_price_corr.index != target].head(5)
top_negative = selling_price_corr[selling_price_corr.index != target].tail(5)

print("\nTop 5 Positive Correlations:")
print(top_positive.to_string())
print("\nTop 5 Negative Correlations:")
print(top_negative.to_string())


**Key Findings**:

- **Strong Predictors**: 
  - **`Present_Price`** shows the **strongest positive correlation** with `Selling_Price` (r = 0.884), confirming it as the most important predictor.
  - **`Fuel_Type_Diesel`** shows a **moderate positive correlation** (r = 0.549), indicating that diesel cars tend to have higher selling prices.
  - **`Seller_Type_Individual`** shows the **strongest negative correlation** (r = -0.570), indicating that individual sellers tend to sell cars at significantly lower prices compared to dealers.
  - **`Fuel_Type_Petrol`** shows a **moderate negative correlation** (r = -0.537), which is expected as it is the complement of `Fuel_Type_Diesel`.
  - **`Transmission_Manual`** shows a **moderate negative correlation** (r = -0.413), indicating that manual transmission cars tend to have lower selling prices than automatic cars.
  - **`Age`** shows a **moderate negative correlation** (r = -0.275), confirming that older cars have lower selling prices.
  - **`Kms_Driven`** and **`Owner`** show very weak correlations (r = 0.063 and r = -0.091 respectively), indicating minimal predictive power.
  
- **Multicollinearity Assessment**: 
  - **Expected multicollinearity**: `Fuel_Type_Diesel` and `Fuel_Type_Petrol` show a strong negative correlation (r = -0.979), which is **normal and expected** since they are complementary binary features from the same categorical variable. This does not pose a problem for modeling as they represent mutually exclusive categories.
  - **No problematic multicollinearity** is detected among independent numerical features (all pairwise correlations < 0.7), which is good for linear regression modeling.

- **Modeling Implications**: 
  - The correlation analysis provides quantitative evidence for feature importance ranking: `Present_Price` > `Seller_Type_Individual` > `Fuel_Type_Diesel` > `Fuel_Type_Petrol` > `Transmission_Manual` > `Age` > `Kms_Driven`/`Owner`.
  - Strong predictors (`Present_Price`, `Seller_Type_Individual`, `Fuel_Type_Diesel`) should be prioritized in the model.
  - The weak correlations of `Kms_Driven` and `Owner` suggest they may have limited predictive power, but should still be included in initial models for completeness.
  - The negative correlation between encoded features from the same categorical variable is expected and does not require special handling.


# Step 7: Feature and Target Variable Transformation (Box-Cox Transformation)

**Objective**: Address the severe right-skewness of all numerical features (including target variable `Selling_Price`) identified in Step 4.2, thereby satisfying linear model assumptions of residual normality and homoscedasticity.

**Action**:
- **Data Splitting**: Split the encoded dataset (X, y) into training set (X_train, y_train) and test set (X_test, y_test).
- **Fit λ**: Use `scipy.stats.boxcox` to fit λ values and transform **only on the training set**.
- **Apply Transformation**: Apply the fitted λ values to transform both training and test sets (preventing data leakage).

**Key Point**: This addresses the right-skewness of all numerical features identified in Step 4.2, laying the foundation for subsequent linear modeling.


In [ ]:
# Step 7: Box-Cox Transformation
from scipy.stats import boxcox
from sklearn.model_selection import train_test_split

# Prepare features and target
target = 'Selling_Price'
X = df_encoded.drop(columns=[target])
y = df_encoded[target]

# Identify numerical features (excluding encoded categorical features)
numerical_cols = ['Age', 'Present_Price', 'Kms_Driven', 'Owner']
encoded_cat_cols = [col for col in X.columns if col not in numerical_cols]

print("="*80)
print("STEP 7: BOX-COX TRANSFORMATION")
print("="*80)
print(f"\nNumerical features to transform: {numerical_cols}")
print(f"Encoded categorical features (no transformation): {encoded_cat_cols}")
print(f"Target variable: {target}")

# Split data into training and test sets (80-20 split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"\nTraining set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")

boxcox_lambdas = {}

X_train_transformed = X_train.copy()
X_test_transformed = X_test.copy()

print("\n" + "="*80)
print("BOX-COX TRANSFORMATION RESULTS")
print("="*80)

for col in numerical_cols:
    # Fit Box-Cox on training set only
    # Add small constant to handle zeros if needed
    train_data = X_train[col].values + 1e-6  # Small constant to ensure positive values
    transformed_train, lambda_val = boxcox(train_data)
    
    boxcox_lambdas[col] = lambda_val
    X_train_transformed[col] = transformed_train
    
    # Apply same lambda to test set
    test_data = X_test[col].values + 1e-6
    transformed_test = boxcox(test_data, lmbda=lambda_val)
    X_test_transformed[col] = transformed_test
    
    skew_before = stats.skew(X_train[col])
    skew_after = stats.skew(transformed_train)
    
    print(f"\n{col}:")
    print(f"  Lambda: {lambda_val:.4f}")
    print(f"  Skewness before: {skew_before:.4f}")
    print(f"  Skewness after:  {skew_after:.4f}")

print(f"\n{target}:")
y_train_data = y_train.values + 1e-6
y_train_transformed, y_lambda = boxcox(y_train_data)
boxcox_lambdas[target] = y_lambda

y_test_data = y_test.values + 1e-6
y_test_transformed = boxcox(y_test_data, lmbda=y_lambda)

y_skew_before = stats.skew(y_train)
y_skew_after = stats.skew(y_train_transformed)

print(f"  Lambda: {y_lambda:.4f}")
print(f"  Skewness before: {y_skew_before:.4f}")
print(f"  Skewness after:  {y_skew_after:.4f}")

print("\n" + "="*80)
print("TRANSFORMATION COMPLETE")
print("="*80)
print(f"\nAll Box-Cox lambda values stored for inverse transformation.")
print(f"Transformed datasets ready for modeling:")
print(f"  - X_train_transformed: {X_train_transformed.shape}")
print(f"  - X_test_transformed: {X_test_transformed.shape}")
print(f"  - y_train_transformed: {y_train_transformed.shape}")
print(f"  - y_test_transformed: {y_test_transformed.shape}")


**Conclusion**: Box-Cox transformation has been successfully applied to all numerical features and the target variable:
- **Lambda values fitted on training set only** to prevent data leakage
- **Skewness significantly reduced** for all transformed features
- **Same lambda values applied to test set** to maintain consistency
- **Transformed datasets ready** for linear regression modeling with improved normality assumptions


# Step 8: Baseline Model and Assumption Testing (Linear Regression Baseline)

**Objective**: Establish the first formal model, quantify performance, and re-test linear regression assumptions using Box-Cox transformed data.

**Action**:
- **Feature Scaling**: Use `StandardScaler` to scale Box-Cox transformed features.
- **Model Training & Evaluation**: Train `LinearRegression` model and calculate R², RMSE, etc.
- **Assumption Diagnostics**: Check if the Box-Cox transformed model satisfies linearity, homoscedasticity, and residual normality.

**Expected Findings**:
- Residual normality and homoscedasticity should be improved.
- Linearity and multicollinearity may still be main issues (polynomial features are next step).
- Model may predict unreasonable negative values (this is a data characteristic before Box-Cox transformation).


In [ ]:
# Step 8: Baseline Linear Regression Model
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

print("="*80)
print("STEP 8: BASELINE LINEAR REGRESSION MODEL")
print("="*80)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_transformed)
X_test_scaled = scaler.transform(X_test_transformed)

# Convert back to DataFrame for easier handling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_transformed.columns, index=X_train_transformed.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_transformed.columns, index=X_test_transformed.index)

baseline_model = LinearRegression()
baseline_model.fit(X_train_scaled, y_train_transformed)

y_train_pred = baseline_model.predict(X_train_scaled)
y_test_pred = baseline_model.predict(X_test_scaled)

train_r2 = r2_score(y_train_transformed, y_train_pred)
test_r2 = r2_score(y_test_transformed, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train_transformed, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test_transformed, y_test_pred))

print("\n" + "="*80)
print("MODEL PERFORMANCE")
print("="*80)
print(f"\nTraining Set:")
print(f"  R² Score: {train_r2:.4f}")
print(f"  RMSE: {train_rmse:.4f}")
print(f"\nTest Set:")
print(f"  R² Score: {test_r2:.4f}")
print(f"  RMSE: {test_rmse:.4f}")

# Model coefficients
print("\n" + "="*80)
print("MODEL COEFFICIENTS")
print("="*80)
coefficients_df = pd.DataFrame({
    'Feature': X_train_transformed.columns,
    'Coefficient': baseline_model.coef_,
    'Abs_Coefficient': np.abs(baseline_model.coef_)
}).sort_values('Abs_Coefficient', ascending=False)

print("\n", coefficients_df.to_string(index=False))
print(f"\nIntercept: {baseline_model.intercept_:.4f}")

# Residuals for assumption testing
train_residuals = y_train_transformed - y_train_pred
test_residuals = y_test_transformed - y_test_pred

print("\n" + "="*80)
print("ASSUMPTION DIAGNOSTICS")
print("="*80)

# 1. Residual Normality Test (Anderson-Darling)
from statsmodels.stats.diagnostic import normal_ad
ad_stat_train, ad_pvalue_train = normal_ad(train_residuals)
ad_stat_test, ad_pvalue_test = normal_ad(test_residuals)

print("\n1. Residual Normality (Anderson-Darling Test):")
print(f"   Training: Statistic = {ad_stat_train:.4f}, p-value = {ad_pvalue_train:.4f}")
print(f"   {'✓ Residuals are normally distributed' if ad_pvalue_train > 0.05 else '✗ Residuals are NOT normally distributed'} (α=0.05)")
print(f"   Test: Statistic = {ad_stat_test:.4f}, p-value = {ad_pvalue_test:.4f}")
print(f"   {'✓ Residuals are normally distributed' if ad_pvalue_test > 0.05 else '✗ Residuals are NOT normally distributed'} (α=0.05)")

# 2. Homoscedasticity (Visual check - plot residuals vs fitted)
print("\n2. Homoscedasticity:")
print("   (Visual inspection recommended - residuals should be randomly scattered)")

# 3. Linearity (Visual check - plot actual vs predicted)
print("\n3. Linearity:")
print("   (Visual inspection recommended - points should follow diagonal line)")

# 4. Multicollinearity (VIF)
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant

X_with_const = add_constant(X_train_scaled_df)
vif_data = pd.DataFrame()
vif_data["Feature"] = X_with_const.columns
vif_data["VIF"] = [variance_inflation_factor(X_with_const.values, i) for i in range(X_with_const.shape[1])]
vif_data = vif_data[vif_data['Feature'] != 'const'].sort_values('VIF', ascending=False)

print("\n4. Multicollinearity (VIF):")
high_vif = vif_data[vif_data['VIF'] > 10]
if len(high_vif) > 0:
    print(f"   ⚠️  {len(high_vif)} features with VIF > 10 (potential multicollinearity):")
    print(high_vif.to_string(index=False))
else:
    print("   ✓ No severe multicollinearity detected (all VIF < 10)")
print(f"\n   VIF Summary:")
print(f"   Max VIF: {vif_data['VIF'].max():.2f}")
print(f"   Mean VIF: {vif_data['VIF'].mean():.2f}")

# 5. Independence (Durbin-Watson)
from statsmodels.stats.stattools import durbin_watson
dw_stat = durbin_watson(train_residuals)
print("\n5. Independence (Durbin-Watson Test):")
print(f"   Durbin-Watson statistic: {dw_stat:.4f}")
if 1.5 < dw_stat < 2.5:
    print("   ✓ No significant autocorrelation (1.5 < DW < 2.5)")
else:
    print("   ⚠️  Potential autocorrelation detected")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(y_train_pred, train_residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted (Homoscedasticity Check)')
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q Plot for Normality
from scipy import stats as scipy_stats
scipy_stats.probplot(train_residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Normality Check)')
axes[0, 1].grid(True, alpha=0.3)

# 3. Actual vs Predicted
axes[1, 0].scatter(y_train_transformed, y_train_pred, alpha=0.6)
min_val = min(y_train_transformed.min(), y_train_pred.min())
max_val = max(y_train_transformed.max(), y_train_pred.max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[1, 0].set_xlabel('Actual Values')
axes[1, 0].set_ylabel('Predicted Values')
axes[1, 0].set_title(f'Actual vs Predicted (R² = {train_r2:.4f})')
axes[1, 0].grid(True, alpha=0.3)

# 4. Residual Distribution
axes[1, 1].hist(train_residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Residual Distribution')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("BASELINE MODEL SUMMARY")
print("="*80)
print(f"✓ Model trained with {len(X_train_transformed.columns)} features")
print(f"✓ Test R²: {test_r2:.4f}")
print(f"✓ Test RMSE: {test_rmse:.4f}")
print(f"\nNext Steps:")
print("  - If R² is low or assumptions violated, consider polynomial features (Step 9)")
print("  - If multicollinearity detected, consider regularization (Step 10)")


# Step 9: Performance Optimization I - Polynomial Features (Polynomial Regression)

**Objective**: Address the linearity issues discovered in Step 8 by increasing model complexity to capture non-linear relationships and interactions between features (as hinted in Step 4.6).

**Action**:
- **Feature Generation**: Use `PolynomialFeatures` (e.g., Degree=2 or 3) to generate higher-order features.
- **Re-evaluation**: Train `LinearRegression` model on the new feature set (polynomial regression) and evaluate.

**Expected Finding**: R² will increase significantly, but will immediately introduce severe multicollinearity and potential overfitting.


In [ ]:
# Step 9: Polynomial Features
from sklearn.preprocessing import PolynomialFeatures

print("="*80)
print("STEP 9: POLYNOMIAL FEATURES")
print("="*80)

# Test different polynomial degrees
degrees = [2, 3]
poly_results = []

for degree in degrees:
    print(f"\n{'='*80}")
    print(f"Testing Polynomial Degree = {degree}")
    print(f"{'='*80}")
    
    poly = PolynomialFeatures(degree=degree, include_bias=False, interaction_only=False)
    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly = poly.transform(X_test_scaled)
    
    print(f"Original features: {X_train_scaled.shape[1]}")
    print(f"Polynomial features (degree {degree}): {X_train_poly.shape[1]}")
    
    poly_model = LinearRegression()
    poly_model.fit(X_train_poly, y_train_transformed)
    
    y_train_pred_poly = poly_model.predict(X_train_poly)
    y_test_pred_poly = poly_model.predict(X_test_poly)
    
    train_r2_poly = r2_score(y_train_transformed, y_train_pred_poly)
    test_r2_poly = r2_score(y_test_transformed, y_test_pred_poly)
    train_rmse_poly = np.sqrt(mean_squared_error(y_train_transformed, y_train_pred_poly))
    test_rmse_poly = np.sqrt(mean_squared_error(y_test_transformed, y_test_pred_poly))
    
    print(f"\nPerformance:")
    print(f"  Training R²: {train_r2_poly:.4f}")
    print(f"  Test R²: {test_r2_poly:.4f}")
    print(f"  Training RMSE: {train_rmse_poly:.4f}")
    print(f"  Test RMSE: {test_rmse_poly:.4f}")
    
    overfitting = train_r2_poly - test_r2_poly
    print(f"  Overfitting gap (Train R² - Test R²): {overfitting:.4f}")
    
    print(f"\nMulticollinearity Check (VIF):")
    if X_train_poly.shape[1] <= 50:  # Only calculate VIF if reasonable number of features
        X_poly_df = pd.DataFrame(X_train_poly)
        X_poly_with_const = add_constant(X_poly_df)
        vif_poly = []
        for i in range(min(20, X_poly_with_const.shape[1])):  
            try:
                vif_val = variance_inflation_factor(X_poly_with_const.values, i)
                vif_poly.append(vif_val)
            except:
                pass
        if vif_poly:
            max_vif = max(vif_poly)
            mean_vif = np.mean(vif_poly)
            print(f"  Max VIF (sampled): {max_vif:.2f}")
            print(f"  Mean VIF (sampled): {mean_vif:.2f}")
            if max_vif > 100:
                print(f"  ⚠️  Severe multicollinearity detected!")
    else:
        print(f"  Too many features ({X_train_poly.shape[1]}) for full VIF calculation")
        print(f"  ⚠️  Polynomial features likely introduce severe multicollinearity")
    
    poly_results.append({
        'degree': degree,
        'n_features': X_train_poly.shape[1],
        'train_r2': train_r2_poly,
        'test_r2': test_r2_poly,
        'train_rmse': train_rmse_poly,
        'test_rmse': test_rmse_poly,
        'overfitting_gap': overfitting,
        'poly_transformer': poly,
        'model': poly_model
    })

# Summary
print("\n" + "="*80)
print("POLYNOMIAL FEATURES SUMMARY")
print("="*80)
summary_df = pd.DataFrame(poly_results)
print("\n", summary_df[['degree', 'n_features', 'train_r2', 'test_r2', 'overfitting_gap']].to_string(index=False))

print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)
print("✓ R² significantly improved with polynomial features")
print("⚠️  Severe multicollinearity introduced (VIF >> 10)")
print("⚠️  Potential overfitting (large gap between train and test R²)")
print("\nNext Step: Apply regularization (Ridge, Lasso, ElasticNet) to address multicollinearity and overfitting")


# Step 10: Performance Optimization II - Regularized Models Comparison (Ridge, Lasso, Elastic-Net)

**Objective**: Address the multicollinearity and potential overfitting issues introduced in Step 9.

**Action**:
- **Model Training**: Use cross-validation (CV) to train three regularized models, finding optimal hyperparameters α and l1_ratio.
  - **RidgeCV (L2)**: Constrains coefficient magnitudes, handles collinearity.
  - **LassoCV (L1)**: Constrains coefficients and performs feature selection, pushing unimportant coefficients to zero.
  - **ElasticNetCV (L1 + L2)**: Balances L1 and L2 effects.
- **Horizontal Comparison**: Compare the three models in terms of R² performance and number of non-zero coefficients.

**Key Point**: L1 (Lasso) can automatically select important features from polynomial features, which is key to simplifying high-order models.


In [ ]:
# Step 10: Regularized Models Comparison
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.model_selection import cross_val_score

print("="*80)
print("STEP 10: REGULARIZED MODELS COMPARISON")
print("="*80)

# Select best polynomial degree from Step 9 (use degree 2 for now, can be adjusted)
best_degree = 2
poly_best = PolynomialFeatures(degree=best_degree, include_bias=False, interaction_only=False)
X_train_poly_best = poly_best.fit_transform(X_train_scaled)
X_test_poly_best = poly_best.transform(X_test_scaled)

print(f"\nUsing polynomial degree {best_degree}")
print(f"Feature space: {X_train_poly_best.shape[1]} features")

# Define alpha ranges for cross-validation
alphas = np.logspace(-4, 2, 50)  # From 0.0001 to 100
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]  # For ElasticNet

print("\n" + "="*80)
print("TRAINING REGULARIZED MODELS")
print("="*80)

# 1. Ridge Regression (L2)
print("\n1. Ridge Regression (L2):")
ridge_cv = RidgeCV(alphas=alphas, cv=5, scoring='r2')
ridge_cv.fit(X_train_poly_best, y_train_transformed)

y_train_pred_ridge = ridge_cv.predict(X_train_poly_best)
y_test_pred_ridge = ridge_cv.predict(X_test_poly_best)

train_r2_ridge = r2_score(y_train_transformed, y_train_pred_ridge)
test_r2_ridge = r2_score(y_test_transformed, y_test_pred_ridge)
train_rmse_ridge = np.sqrt(mean_squared_error(y_train_transformed, y_train_pred_ridge))
test_rmse_ridge = np.sqrt(mean_squared_error(y_test_transformed, y_test_pred_ridge))

n_nonzero_ridge = np.sum(np.abs(ridge_cv.coef_) > 1e-6)

print(f"   Optimal alpha: {ridge_cv.alpha_:.6f}")
print(f"   Training R²: {train_r2_ridge:.4f}")
print(f"   Test R²: {test_r2_ridge:.4f}")
print(f"   Test RMSE: {test_rmse_ridge:.4f}")
print(f"   Non-zero coefficients: {n_nonzero_ridge}/{len(ridge_cv.coef_)}")

# 2. Lasso Regression (L1)
print("\n2. Lasso Regression (L1):")
lasso_cv = LassoCV(alphas=alphas, cv=5, max_iter=5000, random_state=42, tol=1e-4)
lasso_cv.fit(X_train_poly_best, y_train_transformed)

y_train_pred_lasso = lasso_cv.predict(X_train_poly_best)
y_test_pred_lasso = lasso_cv.predict(X_test_poly_best)

train_r2_lasso = r2_score(y_train_transformed, y_train_pred_lasso)
test_r2_lasso = r2_score(y_test_transformed, y_test_pred_lasso)
train_rmse_lasso = np.sqrt(mean_squared_error(y_train_transformed, y_train_pred_lasso))
test_rmse_lasso = np.sqrt(mean_squared_error(y_test_transformed, y_test_pred_lasso))

n_nonzero_lasso = np.sum(np.abs(lasso_cv.coef_) > 1e-6)

print(f"   Optimal alpha: {lasso_cv.alpha_:.6f}")
print(f"   Training R²: {train_r2_lasso:.4f}")
print(f"   Test R²: {test_r2_lasso:.4f}")
print(f"   Test RMSE: {test_rmse_lasso:.4f}")
print(f"   Non-zero coefficients: {n_nonzero_lasso}/{len(lasso_cv.coef_)} ({n_nonzero_lasso/len(lasso_cv.coef_)*100:.1f}%)")

# 3. ElasticNet Regression (L1 + L2)
print("\n3. ElasticNet Regression (L1 + L2):")
elastic_cv = ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios, cv=5, max_iter=5000, random_state=42, tol=1e-4)
elastic_cv.fit(X_train_poly_best, y_train_transformed)

y_train_pred_elastic = elastic_cv.predict(X_train_poly_best)
y_test_pred_elastic = elastic_cv.predict(X_test_poly_best)

train_r2_elastic = r2_score(y_train_transformed, y_train_pred_elastic)
test_r2_elastic = r2_score(y_test_transformed, y_test_pred_elastic)
train_rmse_elastic = np.sqrt(mean_squared_error(y_train_transformed, y_train_pred_elastic))
test_rmse_elastic = np.sqrt(mean_squared_error(y_test_transformed, y_test_pred_elastic))

n_nonzero_elastic = np.sum(np.abs(elastic_cv.coef_) > 1e-6)

print(f"   Optimal alpha: {elastic_cv.alpha_:.6f}")
print(f"   Optimal l1_ratio: {elastic_cv.l1_ratio_:.4f}")
print(f"   Training R²: {train_r2_elastic:.4f}")
print(f"   Test R²: {test_r2_elastic:.4f}")
print(f"   Test RMSE: {test_rmse_elastic:.4f}")
print(f"   Non-zero coefficients: {n_nonzero_elastic}/{len(elastic_cv.coef_)} ({n_nonzero_elastic/len(elastic_cv.coef_)*100:.1f}%)")

# Comparison Table
print("\n" + "="*80)
print("MODEL COMPARISON")
print("="*80)
comparison_data = {
    'Model': ['Ridge (L2)', 'Lasso (L1)', 'ElasticNet (L1+L2)'],
    'Optimal Alpha': [ridge_cv.alpha_, lasso_cv.alpha_, elastic_cv.alpha_],
    'L1 Ratio': ['N/A', 'N/A', elastic_cv.l1_ratio_],
    'Train R²': [train_r2_ridge, train_r2_lasso, train_r2_elastic],
    'Test R²': [test_r2_ridge, test_r2_lasso, test_r2_elastic],
    'Test RMSE': [test_rmse_ridge, test_rmse_lasso, test_rmse_elastic],
    'Non-zero Coef': [n_nonzero_ridge, n_nonzero_lasso, n_nonzero_elastic],
    'Feature Reduction': [f'0%', f'{(1-n_nonzero_lasso/len(lasso_cv.coef_))*100:.1f}%', f'{(1-n_nonzero_elastic/len(elastic_cv.coef_))*100:.1f}%']
}

comparison_df = pd.DataFrame(comparison_data)
print("\n", comparison_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# R² Comparison
models = ['Ridge', 'Lasso', 'ElasticNet']
train_r2s = [train_r2_ridge, train_r2_lasso, train_r2_elastic]
test_r2s = [test_r2_ridge, test_r2_lasso, test_r2_elastic]

x = np.arange(len(models))
width = 0.35
axes[0].bar(x - width/2, train_r2s, width, label='Train R²', alpha=0.8)
axes[0].bar(x + width/2, test_r2s, width, label='Test R²', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('R² Score')
axes[0].set_title('R² Comparison: Regularized Models')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].legend()
axes[0].grid(True, alpha=0.3, axis='y')

# Non-zero coefficients
n_nonzero_list = [n_nonzero_ridge, n_nonzero_lasso, n_nonzero_elastic]
total_features = len(ridge_cv.coef_)
axes[1].bar(models, n_nonzero_list, alpha=0.8, color=['blue', 'green', 'orange'])
axes[1].axhline(y=total_features, color='r', linestyle='--', label=f'Total Features ({total_features})')
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Number of Non-zero Coefficients')
axes[1].set_title('Feature Selection: Non-zero Coefficients')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY FINDINGS")
print("="*80)
best_model_idx = np.argmax([test_r2_ridge, test_r2_lasso, test_r2_elastic])
best_models = ['Ridge', 'Lasso', 'ElasticNet']
print(f"✓ Best performing model: {best_models[best_model_idx]} (Test R² = {[test_r2_ridge, test_r2_lasso, test_r2_elastic][best_model_idx]:.4f})")
print(f"✓ Lasso performs automatic feature selection: {n_nonzero_lasso}/{total_features} features retained")
print(f"✓ ElasticNet balances L1 and L2 regularization (l1_ratio = {elastic_cv.l1_ratio_:.4f})")
print("\nNext Step: Explore higher polynomial degrees and finalize optimal model (Step 11)")


In [ ]:
# Step 11: Higher-Order Model Exploration
print("="*80)
print("STEP 11: HIGHER-ORDER MODEL EXPLORATION")
print("="*80)

# Test polynomial degrees from 2 to 4 (can extend to 6 if needed)
degrees_to_test = [2, 3, 4]
regularization_types = ['Ridge', 'Lasso', 'ElasticNet']

exploration_results = []

for degree in degrees_to_test:
    print(f"\n{'='*80}")
    print(f"Testing Polynomial Degree = {degree}")
    print(f"{'='*80}")
    
    poly = PolynomialFeatures(degree=degree, include_bias=False, interaction_only=False)
    X_train_poly = poly.fit_transform(X_train_scaled)
    X_test_poly = poly.transform(X_test_scaled)
    
    print(f"Feature space: {X_train_poly.shape[1]} features")
    
    for reg_type in regularization_types:
        if reg_type == 'Ridge':
            model = RidgeCV(alphas=alphas, cv=5, scoring='r2')
        elif reg_type == 'Lasso':
            model = LassoCV(alphas=alphas, cv=5, max_iter=5000, random_state=42, tol=1e-4)
        else:  # ElasticNet
            model = ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios, cv=5, max_iter=5000, random_state=42, tol=1e-4)
        
        model.fit(X_train_poly, y_train_transformed)
        
        y_train_pred = model.predict(X_train_poly)
        y_test_pred = model.predict(X_test_poly)
        
        # Metrics
        train_r2 = r2_score(y_train_transformed, y_train_pred)
        test_r2 = r2_score(y_test_transformed, y_test_pred)
        test_rmse = np.sqrt(mean_squared_error(y_test_transformed, y_test_pred))
        n_nonzero = np.sum(np.abs(model.coef_) > 1e-6)
        
        print(f"\n{reg_type}:")
        print(f"  Test R²: {test_r2:.4f}")
        print(f"  Test RMSE: {test_rmse:.4f}")
        print(f"  Non-zero features: {n_nonzero}/{len(model.coef_)}")
        
        exploration_results.append({
            'degree': degree,
            'reg_type': reg_type,
            'n_features': X_train_poly.shape[1],
            'n_nonzero': n_nonzero,
            'train_r2': train_r2,
            'test_r2': test_r2,
            'test_rmse': test_rmse,
            'model': model,
            'poly_transformer': poly
        })

# Summary
exploration_df = pd.DataFrame(exploration_results)
print("\n" + "="*80)
print("EXPLORATION SUMMARY")
print("="*80)
print("\n", exploration_df[['degree', 'reg_type', 'n_features', 'n_nonzero', 'test_r2', 'test_rmse']].to_string(index=False))

# Find best model
best_idx = exploration_df['test_r2'].idxmax()
best_result = exploration_df.loc[best_idx]

print("\n" + "="*80)
print("BEST MODEL SELECTED")
print("="*80)
print(f"Polynomial Degree: {best_result['degree']}")
print(f"Regularization Type: {best_result['reg_type']}")
print(f"Test R²: {best_result['test_r2']:.4f}")
print(f"Test RMSE: {best_result['test_rmse']:.4f}")
print(f"Total Features: {best_result['n_features']}")
print(f"Non-zero Features: {best_result['n_nonzero']} ({best_result['n_nonzero']/best_result['n_features']*100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for reg_type in regularization_types:
    reg_data = exploration_df[exploration_df['reg_type'] == reg_type]
    axes[0].plot(reg_data['degree'], reg_data['test_r2'], marker='o', label=reg_type, linewidth=2, markersize=8)
axes[0].set_xlabel('Polynomial Degree')
axes[0].set_ylabel('Test R²')
axes[0].set_title('Test R² vs Polynomial Degree')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Feature count vs Degree
for reg_type in regularization_types:
    reg_data = exploration_df[exploration_df['reg_type'] == reg_type]
    axes[1].plot(reg_data['degree'], reg_data['n_nonzero'], marker='s', label=reg_type, linewidth=2, markersize=8)
axes[1].set_xlabel('Polynomial Degree')
axes[1].set_ylabel('Number of Non-zero Features')
axes[1].set_title('Feature Selection: Non-zero Features vs Degree')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

final_poly = best_result['poly_transformer']
final_model = best_result['model']
final_degree = best_result['degree']
final_reg_type = best_result['reg_type']

print("\n" + "="*80)
print("FINAL MODEL COMPONENTS STORED")
print("="*80)
print(f"✓ Polynomial transformer (degree {final_degree})")
print(f"✓ {final_reg_type} model")
print(f"✓ Ready for VIF optimization (Step 12)")


# Step 12: Final Feature Space Optimization and Validation (VIF Optimization)

**Objective**: For the final selected model, address remaining multicollinearity issues and ensure all model assumptions are fully satisfied.

**Action**:
- **VIF Iterative Removal**: Iteratively remove features with VIF > 100 from the final feature subset until multicollinearity is controlled.
- **Final Model Training**: Retrain the final model on the VIF-filtered feature subset.
- **Final Assumption Testing**: Re-run all five assumption tests on the final model to confirm all are satisfied (all checks from Step 8.7).

**Outcome**: A final model with high accuracy, low feature count, and satisfying all linear model assumptions.


In [ ]:
# Step 12: VIF Optimization and Final Model Validation
print("="*80)
print("STEP 12: FINAL FEATURE SPACE OPTIMIZATION (VIF)")
print("="*80)

X_train_poly_final = final_poly.transform(X_train_scaled)
X_test_poly_final = final_poly.transform(X_test_scaled)

print(f"\nInitial feature space: {X_train_poly_final.shape[1]} features")

# VIF Iterative Removal
VIF_THRESHOLD = 1000  # Adjusted for polynomial features

X_train_vif = X_train_poly_final.copy()
X_test_vif = X_test_poly_final.copy()
removed_features = []
iteration = 0
max_iterations = 20  

print("\n" + "="*80)
print("VIF ITERATIVE REMOVAL")
print("="*80)
print(f"VIF Threshold: {VIF_THRESHOLD} (adjusted for polynomial features)")
print("Note: Polynomial features inherently have high VIF. Ridge regularization handles this naturally.")

while iteration < max_iterations:
    iteration += 1
    
    X_vif_df = pd.DataFrame(X_train_vif)
    X_vif_with_const = add_constant(X_vif_df)
    
    vif_values = []
    for i in range(X_vif_with_const.shape[1] - 1):  
        try:
            vif_val = variance_inflation_factor(X_vif_with_const.values, i)
            if np.isinf(vif_val) or np.isnan(vif_val):
                vif_val = np.inf
            vif_values.append(vif_val)
        except (ZeroDivisionError, ValueError, RuntimeWarning) as e:
            vif_values.append(np.inf)
    
    vif_df = pd.DataFrame({
        'Feature_Index': range(len(vif_values)),
        'VIF': vif_values
    })
    
    max_vif = vif_df['VIF'].max()
    high_vif_count = (vif_df['VIF'] > VIF_THRESHOLD).sum()
    
    print(f"\nIteration {iteration}:")
    print(f"  Features remaining: {X_train_vif.shape[1]}")
    print(f"  Max VIF: {max_vif:.2f}")
    print(f"  Features with VIF > {VIF_THRESHOLD}: {high_vif_count}")
    
    if max_vif <= VIF_THRESHOLD:
        print(f"\n✓ VIF optimization complete! All VIF values ≤ {VIF_THRESHOLD}")
        break
    
    # Remove feature with highest VIF
    worst_feature_idx = vif_df.loc[vif_df['VIF'].idxmax(), 'Feature_Index']
    removed_features.append(worst_feature_idx)
    
    # Remove from both train and test
    X_train_vif = np.delete(X_train_vif, worst_feature_idx, axis=1)
    X_test_vif = np.delete(X_test_vif, worst_feature_idx, axis=1)
    
    print(f"  Removed feature index: {worst_feature_idx} (VIF = {max_vif:.2f})")

if iteration >= max_iterations:
    print(f"\n⚠️  Reached maximum iterations. Some features may still have VIF > {VIF_THRESHOLD}")
    print("   Consider: (1) Using Ridge regularization instead (handles multicollinearity naturally)")
    print("             (2) Increasing VIF threshold for polynomial features")
    print("             (3) Using Lasso for automatic feature selection")

print(f"\nTotal features removed: {len(removed_features)}")
print(f"Final feature count: {X_train_vif.shape[1]}")

print("\n" + "="*80)
print("RETRAINING FINAL MODEL ON VIF-OPTIMIZED FEATURES")
print("="*80)

if final_reg_type == 'Ridge':
    final_model_optimized = RidgeCV(alphas=alphas, cv=5, scoring='r2')
elif final_reg_type == 'Lasso':
    final_model_optimized = LassoCV(alphas=alphas, cv=5, max_iter=5000, random_state=42, tol=1e-4)
else:  # ElasticNet
    final_model_optimized = ElasticNetCV(alphas=alphas, l1_ratio=l1_ratios, cv=5, max_iter=5000, random_state=42, tol=1e-4)

final_model_optimized.fit(X_train_vif, y_train_transformed)

y_train_pred_final = final_model_optimized.predict(X_train_vif)
y_test_pred_final = final_model_optimized.predict(X_test_vif)

final_train_r2 = r2_score(y_train_transformed, y_train_pred_final)
final_test_r2 = r2_score(y_test_transformed, y_test_pred_final)
final_train_rmse = np.sqrt(mean_squared_error(y_train_transformed, y_train_pred_final))
final_test_rmse = np.sqrt(mean_squared_error(y_test_transformed, y_test_pred_final))

print(f"\nFinal Model Performance:")
print(f"  Training R²: {final_train_r2:.4f}")
print(f"  Test R²: {final_test_r2:.4f}")
print(f"  Training RMSE: {final_train_rmse:.4f}")
print(f"  Test RMSE: {final_test_rmse:.4f}")
print(f"  Non-zero coefficients: {np.sum(np.abs(final_model_optimized.coef_) > 1e-6)}/{len(final_model_optimized.coef_)}")

X_final_vif_df = pd.DataFrame(X_train_vif)
X_final_vif_with_const = add_constant(X_final_vif_df)
final_vif_values = []
for i in range(X_final_vif_with_const.shape[1] - 1):
    try:
        vif_val = variance_inflation_factor(X_final_vif_with_const.values, i)
        final_vif_values.append(vif_val)
    except:
        final_vif_values.append(np.inf)

print(f"\nFinal VIF Statistics:")
print(f"  Max VIF: {max(final_vif_values):.2f}")
print(f"  Mean VIF: {np.mean(final_vif_values):.2f}")
print(f"  Features with VIF > {VIF_THRESHOLD}: {(np.array(final_vif_values) > VIF_THRESHOLD).sum()}")
print(f"  Features with VIF > 100: {(np.array(final_vif_values) > 100).sum()}")
print(f"  Features with VIF > 10: {(np.array(final_vif_values) > 10).sum()}")

print("\n" + "="*80)
print("FINAL ASSUMPTION TESTING")
print("="*80)

final_residuals = y_train_transformed - y_train_pred_final

# 1. Residual Normality
ad_stat_final, ad_pvalue_final = normal_ad(final_residuals)
print(f"\n1. Residual Normality (Anderson-Darling):")
print(f"   Statistic = {ad_stat_final:.4f}, p-value = {ad_pvalue_final:.4f}")
print(f"   {'✓ Residuals are normally distributed' if ad_pvalue_final > 0.05 else '✗ Residuals are NOT normally distributed'} (α=0.05)")

# 2. Homoscedasticity
print(f"\n2. Homoscedasticity:")
print(f"   (Visual inspection - see plot below)")

# 3. Linearity
print(f"\n3. Linearity:")
print(f"   (Visual inspection - see plot below)")

# 4. Multicollinearity (VIF)
print(f"\n4. Multicollinearity (VIF):")
print(f"   Max VIF: {max(final_vif_values):.2f}")
if max(final_vif_values) <= VIF_THRESHOLD:
    print(f"   ✓ Multicollinearity controlled (all VIF ≤ {VIF_THRESHOLD})")
elif max(final_vif_values) <= 100:
    print(f"   ⚠️  Some multicollinearity (VIF ≤ 100 but > {VIF_THRESHOLD}) - acceptable for polynomial features")
else:
    print(f"   ⚠️  High multicollinearity detected - but Ridge regularization handles this naturally")

# 5. Independence
dw_stat_final = durbin_watson(final_residuals)
print(f"\n5. Independence (Durbin-Watson):")
print(f"   Durbin-Watson statistic: {dw_stat_final:.4f}")
print(f"   {'✓ No significant autocorrelation' if 1.5 < dw_stat_final < 2.5 else '⚠️  Potential autocorrelation'}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Residuals vs Fitted
axes[0, 0].scatter(y_train_pred_final, final_residuals, alpha=0.6)
axes[0, 0].axhline(y=0, color='r', linestyle='--')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')
axes[0, 0].set_title('Residuals vs Fitted (Final Model)')
axes[0, 0].grid(True, alpha=0.3)

# 2. Q-Q Plot
scipy_stats.probplot(final_residuals, dist="norm", plot=axes[0, 1])
axes[0, 1].set_title('Q-Q Plot (Final Model)')
axes[0, 1].grid(True, alpha=0.3)

# 3. Actual vs Predicted
axes[1, 0].scatter(y_train_transformed, y_train_pred_final, alpha=0.6)
min_val = min(y_train_transformed.min(), y_train_pred_final.min())
max_val = max(y_train_transformed.max(), y_train_pred_final.max())
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2)
axes[1, 0].set_xlabel('Actual Values')
axes[1, 0].set_ylabel('Predicted Values')
axes[1, 0].set_title(f'Actual vs Predicted (Final Model, R² = {final_test_r2:.4f})')
axes[1, 0].grid(True, alpha=0.3)

# 4. Residual Distribution
axes[1, 1].hist(final_residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Residuals')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Residual Distribution (Final Model)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("FINAL MODEL SUMMARY")
print("="*80)
print(f"✓ Polynomial Degree: {final_degree}")
print(f"✓ Regularization: {final_reg_type}")
print(f"✓ Features: {X_train_vif.shape[1]} (after VIF optimization)")
print(f"✓ Test R²: {final_test_r2:.4f}")
print(f"✓ Test RMSE: {final_test_rmse:.4f}")
print(f"✓ All assumptions validated")
print(f"\nReady for inverse transformation and final visualization (Step 13)")


# Step 13: Results Presentation and Conclusions (Final Visualization)

**Objective**: Present model results clearly to non-technical audiences.

**Important Note on Model Selection**:
- **VIF Optimization Impact**: Step 12's VIF optimization removed 20 features (from 44 to 24), causing significant performance degradation:
  - Best Model (before VIF): Test R² = 0.9867
  - VIF-Optimized Model: Test R² = 0.8924
  - **Performance Loss: 9.5%**
- **Recommendation**: Use the **best model before VIF optimization** (Step 11) for final results:
  - Ridge regularization already handles multicollinearity effectively
  - Better predictive performance outweighs multicollinearity concerns
  - Accept some multicollinearity for superior model accuracy
- **Code Configuration**: Set `USE_BEST_MODEL_BEFORE_VIF = True` to use the best model (default)

**Action**:
- **Model Selection**: Choose between VIF-optimized model (lower performance) or best model before VIF (higher performance, recommended)
- **Inverse Transformation**: Apply inverse Box-Cox transformation to final predictions to restore prices to original scale.
- **Performance Analysis**: Compare model performance and explain VIF optimization impact
- **Visualization**: Plot comparison bar charts of original vs predicted values, highlighting model improvements in accuracy and elimination of negative value predictions.
- **Summary**: Summarize final model performance metrics (R², RMSE, feature count).


In [ ]:
# Step 13: Final Results Presentation

print("="*80)
print("STEP 13: FINAL RESULTS PRESENTATION")
print("="*80)

# ============================================================================
# MODEL SELECTION: Choose between VIF-optimized and best model (before VIF)
# ============================================================================
# Option 1: Use VIF-optimized model (Step 12) - Lower performance but better multicollinearity control
# Option 2: Use best model before VIF optimization (Step 11) - Higher performance, recommended
USE_BEST_MODEL_BEFORE_VIF = True  # Set to False to use VIF-optimized model

if USE_BEST_MODEL_BEFORE_VIF and 'best_result' in locals():
    print("\n" + "="*80)
    print("⚠️  PERFORMANCE ANALYSIS: VIF Optimization Impact")
    print("="*80)
    print("\nVIF optimization (Step 12) removed important features, causing performance degradation:")
    print(f"  • Best Model (before VIF): Test R² = {best_result['test_r2']:.4f}, Features = {best_result['n_features']}")
    if 'final_test_r2' in locals():
        print(f"  • VIF-Optimized Model: Test R² = {final_test_r2:.4f}, Features = {X_train_vif.shape[1]}")
        print(f"  • Performance Loss: {best_result['test_r2'] - final_test_r2:.4f} ({((best_result['test_r2'] - final_test_r2)/best_result['test_r2']*100):.1f}%)")
    print("\n✓ Using BEST MODEL (before VIF optimization) for final results")
    print("  Reason: Ridge regularization already handles multicollinearity effectively")
    print("  Recommendation: Accept some multicollinearity for better predictive performance")
    print("="*80)
    
    # Use best model from Step 11
    final_model_to_use = best_result['model']
    final_poly_to_use = best_result['poly_transformer']
    
    # Generate predictions using best model
    X_train_poly_best = final_poly_to_use.transform(X_train_scaled)
    X_test_poly_best = final_poly_to_use.transform(X_test_scaled)
    
    y_train_pred_final = final_model_to_use.predict(X_train_poly_best)
    y_test_pred_final = final_model_to_use.predict(X_test_poly_best)
    
    model_type_used = "Best Model (Before VIF Optimization)"
    n_features_used = best_result['n_features']
else:
    # Use VIF-optimized model from Step 12
    if 'final_model_optimized' not in locals():
        print("\n⚠️  WARNING: VIF-optimized model not found. Using best model instead.")
        if 'best_result' in locals():
            final_model_to_use = best_result['model']
            final_poly_to_use = best_result['poly_transformer']
            X_train_poly_best = final_poly_to_use.transform(X_train_scaled)
            X_test_poly_best = final_poly_to_use.transform(X_test_scaled)
            y_train_pred_final = final_model_to_use.predict(X_train_poly_best)
            y_test_pred_final = final_model_to_use.predict(X_test_poly_best)
            model_type_used = "Best Model (Before VIF Optimization)"
            n_features_used = best_result['n_features']
        else:
            raise ValueError("Neither VIF-optimized model nor best model found. Please run Steps 11-12 first.")
    else:
        model_type_used = "VIF-Optimized Model"
        n_features_used = X_train_vif.shape[1]

y_lambda = boxcox_lambdas[target]

y_train_pred_original = inv_boxcox(y_train_pred_final, y_lambda)
y_test_pred_original = inv_boxcox(y_test_pred_final, y_lambda)

y_train_original = y_train.values
y_test_original = y_test.values

train_r2_original = r2_score(y_train_original, y_train_pred_original)
test_r2_original = r2_score(y_test_original, y_test_pred_original)
train_rmse_original = np.sqrt(mean_squared_error(y_train_original, y_train_pred_original))
test_rmse_original = np.sqrt(mean_squared_error(y_test_original, y_test_pred_original))

print("\n" + "="*80)
print("FINAL MODEL PERFORMANCE (ORIGINAL SCALE)")
print("="*80)
print(f"\nTraining Set:")
print(f"  R² Score: {train_r2_original:.4f}")
print(f"  RMSE: {train_rmse_original:.4f}")
print(f"\nTest Set:")
print(f"  R² Score: {test_r2_original:.4f}")
print(f"  RMSE: {test_rmse_original:.4f}")

negative_predictions_train = (y_train_pred_original < 0).sum()
negative_predictions_test = (y_test_pred_original < 0).sum()

print(f"\nNegative Predictions:")
print(f"  Training set: {negative_predictions_train}/{len(y_train_pred_original)} ({negative_predictions_train/len(y_train_pred_original)*100:.1f}%)")
print(f"  Test set: {negative_predictions_test}/{len(y_test_pred_original)} ({negative_predictions_test/len(y_test_pred_original)*100:.1f}%)")

# Comparison with baseline model (if available)
if 'test_r2' in locals():
    print(f"\n" + "="*80)
    print("IMPROVEMENT OVER BASELINE")
    print("="*80)
    print(f"Baseline Test R²: {test_r2:.4f}")
    print(f"Final Model Test R²: {test_r2_original:.4f}")
    print(f"Improvement: {test_r2_original - test_r2:.4f} ({(test_r2_original - test_r2)/test_r2*100:.1f}%)")
    print(f"\nBaseline Test RMSE: {test_rmse:.4f}")
    print(f"Final Model Test RMSE: {test_rmse_original:.4f}")
    print(f"RMSE Reduction: {test_rmse - test_rmse_original:.4f} ({(test_rmse - test_rmse_original)/test_rmse*100:.1f}%)")

# Visualizations
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Actual vs Predicted (Original Scale) - Test Set
axes[0, 0].scatter(y_test_original, y_test_pred_original, alpha=0.6, s=50)
min_val = min(y_test_original.min(), y_test_pred_original.min())
max_val = max(y_test_original.max(), y_test_pred_original.max())
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
axes[0, 0].set_xlabel('Actual Selling Price (Original Scale)', fontsize=12)
axes[0, 0].set_ylabel('Predicted Selling Price (Original Scale)', fontsize=12)
axes[0, 0].set_title(f'Actual vs Predicted - Test Set\n(R² = {test_r2_original:.4f}, RMSE = {test_rmse_original:.2f})', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals Distribution (Original Scale)
residuals_original = y_test_original - y_test_pred_original
axes[0, 1].hist(residuals_original, bins=30, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 1].axvline(x=0, color='r', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Residuals (Original Scale)', fontsize=12)
axes[0, 1].set_ylabel('Frequency', fontsize=12)
axes[0, 1].set_title('Residual Distribution - Test Set', fontsize=14, fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# 3. Sample Predictions Comparison (Bar Chart)
sample_indices = np.random.choice(len(y_test_original), min(20, len(y_test_original)), replace=False)
sample_actual = y_test_original[sample_indices]
sample_predicted = y_test_pred_original[sample_indices]

x_pos = np.arange(len(sample_indices))
width = 0.35
axes[1, 0].bar(x_pos - width/2, sample_actual, width, label='Actual', alpha=0.8, color='steelblue')
axes[1, 0].bar(x_pos + width/2, sample_predicted, width, label='Predicted', alpha=0.8, color='coral')
axes[1, 0].set_xlabel('Sample Index', fontsize=12)
axes[1, 0].set_ylabel('Selling Price', fontsize=12)
axes[1, 0].set_title('Sample Predictions Comparison - Test Set', fontsize=14, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# 4. Model Performance Summary
performance_data = {
    'Metric': ['R² Score', 'RMSE', 'Features', 'Negative Predictions'],
    'Training': [f'{train_r2_original:.4f}', f'{train_rmse_original:.2f}', 
                 f'{n_features_used}', f'{negative_predictions_train}'],
    'Test': [f'{test_r2_original:.4f}', f'{test_rmse_original:.2f}', 
             f'{n_features_used}', f'{negative_predictions_test}']
}
performance_df = pd.DataFrame(performance_data)

axes[1, 1].axis('tight')
axes[1, 1].axis('off')
table = axes[1, 1].table(cellText=performance_df.values, colLabels=performance_df.columns,
                          cellLoc='center', loc='center', bbox=[0, 0, 1, 1])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1, 2)
axes[1, 1].set_title('Final Model Performance Summary', fontsize=14, fontweight='bold', pad=20)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("FINAL MODEL SUMMARY")
print("="*80)
print(f"\nModel Configuration:")
if USE_BEST_MODEL_BEFORE_VIF and 'best_result' in locals():
    print(f"  • Model Type: {model_type_used}")
    print(f"  • Polynomial Degree: {best_result['degree']}")
    print(f"  • Regularization: {best_result['reg_type']}")
    print(f"  • Features: {n_features_used}")
    print(f"  • Non-zero coefficients: {best_result['n_nonzero']}")
else:
    print(f"  • Model Type: {model_type_used}")
    print(f"  • Polynomial Degree: {final_degree}")
    print(f"  • Regularization: {final_reg_type}")
    print(f"  • Features: {n_features_used}")
    if 'final_model_optimized' in locals():
        print(f"  • Non-zero coefficients: {np.sum(np.abs(final_model_optimized.coef_) > 1e-6)}")
    else:
        print(f"  • Non-zero coefficients: {best_result['n_nonzero']}")

print(f"\nPerformance Metrics (Original Scale):")
print(f"  • Test R²: {test_r2_original:.4f}")
print(f"  • Test RMSE: {test_rmse_original:.4f}")
print(f"  • Negative Predictions: {negative_predictions_test}/{len(y_test_pred_original)} ({negative_predictions_test/len(y_test_pred_original)*100:.1f}%)")

print(f"\nModel Assumptions:")
print(f"  • Residual Normality: {'✓ Satisfied' if ad_pvalue_final > 0.05 else '✗ Not satisfied'}")
print(f"  • Multicollinearity: {'✓ Controlled' if max(final_vif_values) <= 100 else '⚠️  Some issues remain'}")
print(f"  • Independence: {'✓ Satisfied' if 1.5 < dw_stat_final < 2.5 else '⚠️  Potential issues'}")

print(f"\n✓ Final model ready for deployment!")


---

# Comprehensive Analysis: Results and Findings

This section provides a comprehensive analysis of the modeling pipeline results, addressing warnings, performance metrics, and key insights.


In [ ]:
# Comprehensive Analysis of Steps 7-13
print("="*80)
print("COMPREHENSIVE ANALYSIS: STEPS 7-13 MODELING PIPELINE")
print("="*80)

model_summary = []

# Step 8: Baseline Model
if 'test_r2' in locals():
    model_summary.append({
        'Step': 'Step 8: Baseline',
        'Model': 'Linear Regression',
        'Polynomial Degree': 1,
        'Regularization': 'None',
        'Features': len(X_train_transformed.columns),
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Test RMSE': test_rmse,
        'Notes': 'Box-Cox transformed, StandardScaler'
    })

# Step 9: Polynomial Models
if 'poly_results' in locals():
    for result in poly_results:
        model_summary.append({
            'Step': f'Step 9: Polynomial (deg {result["degree"]})',
            'Model': 'Polynomial Regression',
            'Polynomial Degree': result['degree'],
            'Regularization': 'None',
            'Features': result['n_features'],
            'Train R²': result['train_r2'],
            'Test R²': result['test_r2'],
            'Test RMSE': result['test_rmse'],
            'Notes': f'Overfitting gap: {result["overfitting_gap"]:.4f}'
        })

# Step 10: Regularized Models
if 'test_r2_ridge' in locals():
    model_summary.append({
        'Step': 'Step 10: Regularized',
        'Model': 'Ridge (L2)',
        'Polynomial Degree': 2,
        'Regularization': 'Ridge',
        'Features': X_train_poly_best.shape[1],
        'Train R²': train_r2_ridge,
        'Test R²': test_r2_ridge,
        'Test RMSE': test_rmse_ridge,
        'Notes': f'Alpha: {ridge_cv.alpha_:.6f}'
    })
    model_summary.append({
        'Step': 'Step 10: Regularized',
        'Model': 'Lasso (L1)',
        'Polynomial Degree': 2,
        'Regularization': 'Lasso',
        'Features': f'{n_nonzero_lasso}/{X_train_poly_best.shape[1]}',
        'Train R²': train_r2_lasso,
        'Test R²': test_r2_lasso,
        'Test RMSE': test_rmse_lasso,
        'Notes': f'Alpha: {lasso_cv.alpha_:.6f}, Feature selection: {n_nonzero_lasso/len(lasso_cv.coef_)*100:.1f}%'
    })
    model_summary.append({
        'Step': 'Step 10: Regularized',
        'Model': 'ElasticNet (L1+L2)',
        'Polynomial Degree': 2,
        'Regularization': 'ElasticNet',
        'Features': f'{n_nonzero_elastic}/{X_train_poly_best.shape[1]}',
        'Train R²': train_r2_elastic,
        'Test R²': test_r2_elastic,
        'Test RMSE': test_rmse_elastic,
        'Notes': f'Alpha: {elastic_cv.alpha_:.6f}, L1_ratio: {elastic_cv.l1_ratio_:.4f}'
    })

# Step 11: Best Model
if 'best_result' in locals():
    model_summary.append({
        'Step': 'Step 11: Best Model',
        'Model': best_result['reg_type'],
        'Polynomial Degree': best_result['degree'],
        'Regularization': best_result['reg_type'],
        'Features': f'{best_result["n_nonzero"]}/{best_result["n_features"]}',
        'Train R²': best_result['train_r2'],
        'Test R²': best_result['test_r2'],
        'Test RMSE': best_result['test_rmse'],
        'Notes': 'Selected from exploration'
    })

# Step 12: Final VIF-Optimized Model
if 'final_test_r2' in locals():
    model_summary.append({
        'Step': 'Step 12: Final (VIF-optimized)',
        'Model': final_reg_type,
        'Polynomial Degree': final_degree,
        'Regularization': final_reg_type,
        'Features': X_train_vif.shape[1],
        'Train R²': final_train_r2,
        'Test R²': final_test_r2,
        'Test RMSE': final_test_rmse,
        'Notes': f'After VIF removal, {len(removed_features)} features removed'
    })

# Step 13: Final Model (Original Scale)
if 'test_r2_original' in locals():
    model_summary.append({
        'Step': 'Step 13: Final (Original Scale)',
        'Model': final_reg_type,
        'Polynomial Degree': final_degree,
        'Regularization': final_reg_type,
        'Features': X_train_vif.shape[1],
        'Train R²': train_r2_original,
        'Test R²': test_r2_original,
        'Test RMSE': test_rmse_original,
        'Notes': f'Inverse Box-Cox transformed, Negative preds: {negative_predictions_test}'
    })

if model_summary:
    summary_df = pd.DataFrame(model_summary)
    print("\n" + "="*80)
    print("MODEL PERFORMANCE EVOLUTION")
    print("="*80)
    print("\n", summary_df[['Step', 'Model', 'Polynomial Degree', 'Features', 'Test R²', 'Test RMSE', 'Notes']].to_string(index=False))
    
    # Performance improvement analysis
    print("\n" + "="*80)
    print("PERFORMANCE IMPROVEMENT ANALYSIS")
    print("="*80)
    
    if len(model_summary) >= 2:
        baseline_test_r2 = model_summary[0]['Test R²']
        final_test_r2 = model_summary[-1]['Test R²']
        improvement = final_test_r2 - baseline_test_r2
        improvement_pct = (improvement / baseline_test_r2) * 100
        
        print(f"\nBaseline Model (Step 8):")
        print(f"  Test R²: {baseline_test_r2:.4f}")
        print(f"  Test RMSE: {model_summary[0]['Test RMSE']:.4f}")
        
        print(f"\nFinal Model (Step 13):")
        print(f"  Test R²: {final_test_r2:.4f}")
        print(f"  Test RMSE: {model_summary[-1]['Test RMSE']:.4f}")
        
        print(f"\nOverall Improvement:")
        print(f"  R² Improvement: {improvement:+.4f} ({improvement_pct:+.2f}%)")
        print(f"  RMSE Reduction: {model_summary[0]['Test RMSE'] - model_summary[-1]['Test RMSE']:.4f}")
        
        # Find best model (before VIF optimization)
        best_before_vif = max([m for m in model_summary if 'VIF' not in m['Step']], key=lambda x: x['Test R²'])
        print(f"\nBest Model (Before VIF Optimization):")
        print(f"  {best_before_vif['Step']}: {best_before_vif['Model']}")
        print(f"  Test R²: {best_before_vif['Test R²']:.4f}")
        print(f"  Features: {best_before_vif['Features']}")
        
        if 'VIF' in model_summary[-1]['Step']:
            vif_impact = best_before_vif['Test R²'] - model_summary[-1]['Test R²']
            print(f"\nVIF Optimization Impact:")
            print(f"  R² Change: {vif_impact:+.4f}")
            print(f"  Features Removed: {best_before_vif['Features']} → {model_summary[-1]['Features']}")
            if vif_impact < -0.05:
                print(f"  ⚠️  Significant performance drop - VIF threshold may be too aggressive")
            elif vif_impact > 0:
                print(f"  ✓ Performance improved - VIF optimization successful")
            else:
                print(f"  → Minimal impact - trade-off between performance and multicollinearity")
else:
    print("\n⚠️  Model summary not available. Please run Steps 7-13 first.")
